# NB10 — Default models retrained with the COA final-training schedule

**Why this notebook exists.** In NB01 the Default ANN / Default CNN+BiLSTM baselines were trained for at most
50 / 20 epochs (early-stopping patience 5 / 3), while NB04 trained the headline COA-ANN / COA-CNN+BiLSTM models for
at most 100 / 30 epochs (patience 10 / 5) with a ModelCheckpoint. The headline COA-vs-Default comparison therefore
mixed two effects: hyperparameters *and* training budget. This notebook retrains the default configurations with
**exactly** the NB04 protocol, so the only difference left is the hyperparameters.

**Inputs (Add Input → Notebook output):**
- `01-preprocessing` (shared_data + baseline metrics)
- `04-final-training` (COA headline metrics)

**Outputs:** `saved_results/{dataset}/default_matched_{ann,cnn}_metrics.json`, `..._prob.npy`,
`saved_results/global/matched_schedule_comparison.csv`.

Accelerator: GPU T4 x2 (same as NB04).

In [ ]:
# ── Identical environment to NB04 ──────────────────────────────────────────────
import os, random as _random, warnings, json, gc
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED       = 42
EMBED_DIM  = 300
DATASETS   = ["amazon", "yelp", "twitter"]

KAGGLE_USER = "abhaysharma01702"
def nb_input(slug):
    return f"/kaggle/input/notebooks/{KAGGLE_USER}/{slug}"

WORKING     = "/kaggle/working"
RESULTS_DIR = f"{WORKING}/saved_results"
MODELS_DIR  = f"{WORKING}/saved_models"
for base in [RESULTS_DIR, MODELS_DIR]:
    for ds in DATASETS + ["global"]:
        os.makedirs(f"{base}/{ds}", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed)
    np.random.seed(seed)

set_seeds()
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight

tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
N_GPUS = strategy.num_replicas_in_sync
ANN_BATCH = 1024 * N_GPUS
CNN_BATCH = 128  * N_GPUS
print(f"MirroredStrategy: {N_GPUS} GPU(s)  ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}  TF {tf.__version__}")

def set_seeds_tf(seed=SEED):
    set_seeds(seed)
    tf.random.set_seed(seed)

set_seeds_tf()

In [ ]:
# ── Helpers copied verbatim from NB04 ─────────────────────────────────────────
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.float32)))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.int32), y.astype(np.float32)))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def build_ann(h1, h2, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_cnn(filters, lstm_units, lr, vocab_sz, emb_mat, max_len, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Embedding(vocab_sz, EMBED_DIM, weights=[emb_mat], trainable=False),
            tf.keras.layers.Conv1D(filters, 3, activation="relu"),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Helpers ready.")

In [ ]:
# ── Linked inputs ─────────────────────────────────────────────────────────────
NB01_SHARED  = f"{nb_input('01-preprocessing')}/shared_data"
NB01_RESULTS = f"{nb_input('01-preprocessing')}/saved_results"
NB04_RESULTS = f"{nb_input('04-final-training')}/saved_results"

for ds in DATASETS:
    ok01 = os.path.exists(f"{NB01_SHARED}/{ds}/config.json")
    ok04 = os.path.exists(f"{NB04_RESULTS}/{ds}/ann_coa_metrics.json")
    print(f"  {ds}: NB01 {'✓' if ok01 else '✗ MISSING'}   NB04 {'✓' if ok04 else '✗ MISSING'}")

def load_arrays(ds):
    d = f"{NB01_SHARED}/{ds}"
    cfg = json.load(open(f"{d}/config.json"))
    out = {k: np.load(f"{d}/{k}.npy") for k in
           ["X_emb_train", "X_emb_val", "X_emb_test", "X_seq_train", "X_seq_val",
            "X_seq_test", "y_train", "y_val", "y_test", "emb_matrix"]}
    out["max_len"] = cfg["max_len"]; out["vocab_size"] = cfg["vocab_size"]
    return out

In [ ]:
# ── Retrain defaults with the NB04 schedule (100/30 epochs, patience 10/5, checkpoint) ──
DEF_ANN = (128, 64, 1e-3)
DEF_CNN = (128, 64, 1e-3)
matched = {}

for ds in DATASETS:
    print(f"\n{'='*60}\nMATCHED-SCHEDULE DEFAULTS | {ds.upper()}\n{'='*60}")
    d = load_arrays(ds)
    X_emb_tv = np.vstack([d["X_emb_train"], d["X_emb_val"]])
    X_seq_tv = np.vstack([d["X_seq_train"], d["X_seq_val"]])
    y_tv     = np.concatenate([d["y_train"], d["y_val"]])
    cw_tv = dict(zip([int(k) for k in np.unique(y_tv)],
                     [float(v) for v in compute_class_weight("balanced", classes=np.unique(y_tv), y=y_tv)]))
    n_es = int(len(y_tv) * 0.1)
    m_dir = f"{MODELS_DIR}/{ds}"; r_dir = f"{RESULTS_DIR}/{ds}"

    # Default ANN — same code path as NB04 COA-ANN
    set_seeds_tf()
    model = build_ann(*DEF_ANN)
    ckpt = f"{m_dir}/default_ann_matched_best.keras"
    model.fit(make_tf_ds(X_emb_tv[:-n_es], y_tv[:-n_es], ANN_BATCH),
              validation_data=make_tf_ds(X_emb_tv[-n_es:], y_tv[-n_es:], ANN_BATCH, shuffle=False),
              epochs=100,
              callbacks=[make_es(10, verbose=1),
                         ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, verbose=0)],
              class_weight=cw_tv, verbose=2)
    model = tf.keras.models.load_model(ckpt)
    p = model.predict(make_tf_ds(d["X_emb_test"], np.zeros(len(d["y_test"])), ANN_BATCH, shuffle=False),
                      verbose=0).flatten()
    ann_res = evaluate("Default ANN (matched schedule)", d["y_test"], (p > 0.5).astype(int), p)
    np.save(f"{r_dir}/default_matched_ann_prob.npy", p)
    json.dump(ann_res, open(f"{r_dir}/default_matched_ann_metrics.json", "w"), indent=2)
    print(f"  Default ANN (matched): F1={ann_res['F1-score']}  MCC={ann_res['MCC']}")
    del model; gc.collect(); tf.keras.backend.clear_session()

    # Default CNN+BiLSTM — same code path as NB04 COA-CNN
    set_seeds_tf()
    model = build_cnn(*DEF_CNN, d["vocab_size"], d["emb_matrix"], d["max_len"])
    ckpt = f"{m_dir}/default_cnn_matched_best.keras"
    model.fit(make_seq_ds(X_seq_tv[:-n_es], y_tv[:-n_es], CNN_BATCH),
              validation_data=make_seq_ds(X_seq_tv[-n_es:], y_tv[-n_es:], CNN_BATCH, shuffle=False),
              epochs=30,
              callbacks=[make_es(5, verbose=1),
                         ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, verbose=0)],
              class_weight=cw_tv, verbose=2)
    model = tf.keras.models.load_model(ckpt)
    p = model.predict(make_seq_ds(d["X_seq_test"], np.zeros(len(d["y_test"])), CNN_BATCH, shuffle=False),
                      verbose=0).flatten()
    cnn_res = evaluate("Default CNN+BiLSTM (matched schedule)", d["y_test"], (p > 0.5).astype(int), p)
    np.save(f"{r_dir}/default_matched_cnn_prob.npy", p)
    json.dump(cnn_res, open(f"{r_dir}/default_matched_cnn_metrics.json", "w"), indent=2)
    print(f"  Default CNN (matched): F1={cnn_res['F1-score']}  MCC={cnn_res['MCC']}")
    del model; gc.collect(); tf.keras.backend.clear_session()

    matched[ds] = {"ann": ann_res, "cnn": cnn_res}
    del d, X_emb_tv, X_seq_tv, y_tv; gc.collect()

print("\nMatched-schedule retraining complete.")

In [ ]:
# ── Comparison: COA (NB04) vs Default matched (NB10) vs Default original (NB01) ──
rows = []
for ds in DATASETS:
    base = json.load(open(f"{NB01_RESULTS}/{ds}/baseline_metrics.json"))
    coa_ann = json.load(open(f"{NB04_RESULTS}/{ds}/ann_coa_metrics.json"))
    coa_cnn = json.load(open(f"{NB04_RESULTS}/{ds}/cnn_coa_metrics.json"))
    for arch, coa, orig_name in [("ANN", coa_ann, "Default ANN"), ("CNN+BiLSTM", coa_cnn, "Default CNN+BiLSTM")]:
        new = matched[ds]["ann" if arch == "ANN" else "cnn"]
        orig = base[orig_name]
        for metric in ["F1-score", "Macro-F1", "MCC", "AUC-ROC"]:
            rows.append({"Dataset": ds.capitalize(), "Architecture": arch, "Metric": metric,
                         "COA (NB04, 100/30 ep)": coa[metric],
                         "Default matched (NB10, 100/30 ep)": new[metric],
                         "Default original (NB01, 50/20 ep)": orig[metric],
                         "COA - Default matched": round(coa[metric] - new[metric], 4)})
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(f"{RESULTS_DIR}/global/matched_schedule_comparison.csv", index=False)
print(cmp_df[cmp_df["Metric"] == "F1-score"].to_string(index=False))